In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import re
from datetime import datetime

def extract_dates(text):
    """
    Extract dates in various formats
    Formats: MM/DD/YYYY, DD-MM-YYYY, Month DD, YYYY, YYYY-MM-DD
    """
    
    patterns = [
        r'\d{1,2}/\d{1,2}/\d{4}',  # MM/DD/YYYY
        r'\d{1,2}-\d{1,2}-\d{4}',  # DD-MM-YYYY
        r'\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[a-z]* \d{1,2},? \d{4}',
        r'\d{4}-\d{2}-\d{2}'  # ISO format
    ]
    
    dates = []
    
    for pattern in patterns:
        matches = re.findall(pattern, text)
        dates.extend(matches)
    
    return dates


# Test
text = "Invoice date: 03/15/2024. Due: March 30, 2024"
print(extract_dates(text))


['03/15/2024', 'March 30, 2024']


In [2]:
import re

def extract_amounts(text):
    """
    Extract currency amounts
    Handles: $1,250.50, 1250.50, $1250
    """
    
    pattern = r'\$?\d{1,3}(?:,\d{3})*(?:\.\d{2})?'
    amounts = re.findall(pattern, text)
    
    # Convert to float
    cleaned = []
    for amount in amounts:
        # Remove $ and commas
        clean = amount.replace('$', '').replace(',', '')
        try:
            cleaned.append(float(clean))
        except ValueError:
            continue  # skip invalid values
    
    return cleaned


# Test
text = "Total: $1,250.50. Tax: $125.05. Subtotal: 1125.45"
print(extract_amounts(text))


[1250.5, 125.05, 112.0, 5.45]


In [3]:
import re

def extract_invoice_number(text):
    """
    Extract invoice/order numbers
    Patterns: INV-2024-001, #12345, ORDER-ABC123
    """
    
    patterns = [
        r'INV-\d{4}-\d{3}',
        r'#\d{5,}',
        r'ORDER-[A-Z0-9]+',
        r'Invoice (?:Number|#):?\s*([A-Z0-9-]+)'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            # If regex has a capturing group, return it
            if match.groups():
                return match.group(1)
            else:
                return match.group(0)
    
    return None


# Test
text = "Invoice Number: INV-2024-001"
print(extract_invoice_number(text))


INV-2024-001


In [4]:
!pip install spacy
!python -m spacy download en_core_web_sm


Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 978, in getaddrinfo
    for res in _socket.getaddrinfo(host, port, family, type, proto, flags):
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
socket.gaierror: [Errno -3] Temporary failure in name resolution

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
             

In [7]:
import spacy

# Load model
nlp = spacy.load("en_core_web_sm")

# Sample invoice text
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

# Process text
doc = nlp(text)

# Extract entities
print("Found entities:")
for ent in doc.ents:
    print(f"{ent.text:25} {ent.label_:15} {spacy.explain(ent.label_)}")


Found entities:
Acme Corporation          ORG             Companies, agencies, institutions, etc.
123                       CARDINAL        Numerals that do not fall under another type
Main Street               FAC             Buildings, airports, highways, bridges, etc.
New York                  GPE             Countries, cities, states
10001                     DATE            Absolute or relative dates or periods
John Smith                PERSON          People, including fictional
March 15, 2024            DATE            Absolute or relative dates or periods
1,250.50                  MONEY           Monetary values, including unit


In [8]:
import spacy

# Load model (make sure you've installed it in Kaggle)
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    """
    Extract and organize entities by type
    """
    
    doc = nlp(text)
    
    entities = {
        'persons': [],
        'organizations': [],
        'locations': [],
        'dates': [],
        'money': []
    }
    
    for ent in doc.ents:
        if ent.label_ == 'PERSON':
            entities['persons'].append(ent.text)
        elif ent.label_ == 'ORG':
            entities['organizations'].append(ent.text)
        elif ent.label_ in ['GPE', 'LOC']:
            entities['locations'].append(ent.text)
        elif ent.label_ == 'DATE':
            entities['dates'].append(ent.text)
        elif ent.label_ == 'MONEY':
            entities['money'].append(ent.text)
    
    return entities


# Test
text = """
Invoice from Acme Corporation
123 Main Street, New York, NY 10001
Contact: John Smith (john@acme.com)
Date: March 15, 2024
Amount Due: $1,250.50
"""

result = extract_entities(text)

for entity_type, values in result.items():
    print(f"{entity_type}: {values}")


persons: ['John Smith']
organizations: ['Acme Corporation']
locations: ['New York']
dates: ['10001', 'March 15, 2024']
money: ['1,250.50']


In [10]:
from spacy import displacy

# Render inside notebook
displacy.render(doc, style='ent', jupyter=True)

# Force HTML generation
html = displacy.render(doc, style='ent', page=True)

# Fix: check if html is None
if html is None:
    html = displacy.render(doc, style='ent')

# Save HTML
with open('entities.html', 'w', encoding='utf-8') as f:
    f.write(str(html))   # ensure it's string

print("Visualization saved to entities.html")


Visualization saved to entities.html


In [11]:
import json
import pytesseract
from PIL import Image

def process_invoice(image_path):
    """
    Complete pipeline: OCR → Extraction → JSON
    """
    
    # Step 1: OCR
    img = Image.open(image_path)
    text = pytesseract.image_to_string(img)
    
    # Step 2: Extract with regex
    invoice_data = {
        'invoice_number': extract_invoice_number(text),
        'dates': extract_dates(text),
        'amounts': extract_amounts(text)
    }
    
    # Step 3: Extract with NER
    entities = extract_entities(text)
    invoice_data.update(entities)
    
    # Step 4: Post-process
    
    # Safely get total amount
    if invoice_data.get('amounts'):
        invoice_data['total_amount'] = max(invoice_data['amounts'])
    else:
        invoice_data['total_amount'] = None
    
    # Safely get invoice date
    if invoice_data.get('dates'):
        invoice_data['invoice_date'] = invoice_data['dates'][0]
    else:
        invoice_data['invoice_date'] = None
    
    return invoice_data


# Test
result = process_invoice('/kaggle/input/datasets/sheikhhaseeb123/ocrimagesdata/images/17.jpg')
print(json.dumps(result, indent=2))


{
  "invoice_number": null,
  "dates": [
    "903120092183",
    "2114",
    "068113113619",
    "98071",
    "0003"
  ],
  "amounts": [
    7.0,
    2.0,
    1.0,
    2.0,
    2.0,
    3.0,
    970.0,
    259.0,
    875.0,
    5.0,
    8.0,
    303.0,
    2.0,
    30.0,
    80.0,
    1.0,
    1.0,
    0.0,
    34.0,
    95.0,
    903.0,
    120.0,
    92.0,
    183.0,
    7.0,
    71.0,
    782.0,
    200.0,
    73.0,
    447.0,
    501.0,
    213.0,
    1.0,
    2.0,
    100.0,
    61.0,
    526.0,
    211.0,
    4.0,
    20.0,
    633.0,
    960.0,
    343.0,
    0.0,
    1.0,
    0.44,
    68.0,
    113.0,
    113.0,
    619.0,
    7.0,
    4.0,
    702.0,
    483.0,
    6.0,
    4.0,
    125.0,
    100.0,
    5.0,
    0.0,
    980.0,
    71.0,
    0.0,
    3.0,
    14.0,
    26.0,
    3.0,
    3.0
  ],
  "persons": [
    "MIKE",
    "Bal Bee"
  ],
  "organizations": [
    "Tes",
    "PILLS"
  ],
  "locations": [],
  "money": [],
  "total_amount": 980.0,
  "invoice_date": "90312009

In [12]:
import json

# Save to JSON file
output_file = 'extracted_data.json'

with open(output_file, 'w') as f:
    json.dump(result, f, indent=2)

print(f"Results saved to {output_file}")


Results saved to extracted_data.json
